# DS207 Final Project - Predicting Readmission Rate for Diabetic Patients Using Machine Learning
#### Contributor: Rahil Sharma (rahilsharma@berkeley.edu)

### Notebook Structure:

1. Data Processing:
2. Baseline Model: 
    * Majority
    * Multiclass Logistic Regression
3. Notebook Exports:
    * Datasets: `results/train.csv`, `results/val.csv`, and `results/test.csv`
    * Baseline model: in `baseline.pkl`

## 1. Data Processing

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif
import altair as alt
import seaborn as sns
alt.data_transformers.enable('vegafusion')

import joblib

In [2]:
! pip install "vegafusion[embed]>=1.5.0"

In [3]:
! pip install "vl-convert-python>=1.6.0"

In [4]:
df = pd.read_csv('../data/diabetic_data.csv')
raw_df = pd.read_csv('../data/diabetic_data.csv')
display(df.head())

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### Key for Drug descriptions ##

Up: The dosage of the drug was increased for the patient during their encounter.

Down: The dosage was decreased.

Steady: The patient is on the drug, and the dosage was not changed.

No: The patient was not prescribed this specific drug.

In [5]:
print(f"There are {len(df.columns)} columns:\n", df.columns)

There are 50 columns:
 Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')


#### Drop duplicates and unnecessary columns

In [6]:
df = df.drop_duplicates(subset=['patient_nbr'], keep='first')

In [7]:
# checking to see how many missing values there are
(df == '?').sum()

# columns to keep based on descriptions / number of missing values
  # race (split into indicator variables)
  # gender (split into indicator variables)
  # age (split into indicator variables)
  # time_in_hospital (int)
  # num_lab_procedures (int)
  # num_procedures (int)
  # num_medications (int)
  # number_outpatient (int)
  # number_emergency (int)
  # number_inpatient (int)
  # number_diagnoses (int)
  # keeping all the indicator variables for medication / medication changes
  # target - multiclass classification

# dropping the unnecessary columns
df = df.drop(columns=['encounter_id', 'patient_nbr', 'weight', 'admission_type_id',
                      'discharge_disposition_id', 'admission_source_id', 'payer_code',
                      'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
                      'A1Cresult'],
             axis=1)

#### Collapse age and race

In [8]:
# Convert age buckets like "[70-80)" to numeric midpoint (75)
def age_to_mid(age_bucket):
    if pd.isna(age_bucket):
        return np.nan
    try:
        lo, hi = age_bucket.strip('[]').split('-')
        lo = int(lo)
        hi = int(hi.strip(')'))
        return (lo + hi) / 2
    except Exception:
        return np.nan
df['age_mid'] = df['age'].apply(age_to_mid)
df.drop(columns=['age'], inplace=True)

In [9]:
# Collapse low count race categories into 'Other'
if 'race' in df.columns:
    df['race'] = df['race'].fillna('Unknown')
    top = df['race'].value_counts().nlargest(4).index
    df['race_collapsed'] = df['race'].where(df['race'].isin(top), other='Other')
    df.drop(columns=['race'], inplace=True)

#### Categorical Features

In [10]:
# creating indicator variables for the categorical features (one-hot encoding)
df = pd.get_dummies(df, columns=['race_collapsed', 'gender'], dtype=int)

# Encode ordinal values according to the prescription status of the visit
prescription_map = {'No': 0,       # The drug was not prescribed
                    'Down': 1,     # The dosage was decreased
                    'Steady': 2,   # The dosage did not change
                    'Up': 3}       # The dosage was increased during the encounter
cat_columns = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
               'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
               'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
               'miglitol', 'troglitazone', 'tolazamide', 'examide',
               'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
               'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
for cat_col in cat_columns:
    df[cat_col] = df[cat_col].map(prescription_map)

#### Binary features and Outcome of Interest

In [11]:
# displaying the data
print("The number of columns are: ", len(df.columns))
display(df.head())
print(df.columns)

# recoding change and diabetesMed to 0 for No and 1 to Yes
df['change'] = np.where(df['change'] == 'No', 0, 1)
df['diabetesMed'] = np.where(df['diabetesMed'] == 'No', 0, 1)

# mapping the target into three classes
readmission_map = {
    'NO': 0,  # no readmission recorded
    '>30': 1, # readmitted (over 30 days)
    '<30': 1  # readmitted (within 30 days)
}

df['readmitted'] = df['readmitted'].map(readmission_map)

display(df.head())

df['readmitted'].value_counts()


The number of columns are:  43


,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,readmitted,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
0,1,41,0,1,0,0,0,1,0,0,...,NO,5.0,0,0,1,0,0,1,0,0
1,3,59,0,18,0,0,0,9,0,0,...,>30,15.0,0,0,1,0,0,1,0,0
2,2,11,5,13,2,0,1,6,0,0,...,NO,25.0,0,1,0,0,0,1,0,0
3,2,44,1,16,0,0,0,7,0,0,...,NO,35.0,0,0,1,0,0,0,1,0
4,1,51,0,8,0,0,0,5,0,0,...,NO,45.0,0,0,1,0,0,0,1,0


Index(['time_in_hospital', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'number_diagnoses', 'metformin', 'repaglinide',
       'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',
       'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
       'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
       'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
       'glipizide-metformin', 'glimepiride-pioglitazone',
       'metformin-rosiglitazone', 'metformin-pioglitazone', 'change',
       'diabetesMed', 'readmitted', 'age_mid', 'race_collapsed_?',
       'race_collapsed_AfricanAmerican', 'race_collapsed_Caucasian',
       'race_collapsed_Hispanic', 'race_collapsed_Other', 'gender_Female',
       'gender_Male', 'gender_Unknown/Invalid'],
      dtype='object')


,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,readmitted,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
0,1,41,0,1,0,0,0,1,0,0,...,0,5.0,0,0,1,0,0,1,0,0
1,3,59,0,18,0,0,0,9,0,0,...,1,15.0,0,0,1,0,0,1,0,0
2,2,11,5,13,2,0,1,6,0,0,...,0,25.0,0,1,0,0,0,1,0,0
3,2,44,1,16,0,0,0,7,0,0,...,0,35.0,0,0,1,0,0,0,1,0
4,1,51,0,8,0,0,0,5,0,0,...,0,45.0,0,0,1,0,0,0,1,0


readmitted
0    42985
1    28533
Name: count, dtype: int64

In [12]:
df.describe()

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,readmitted,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
count,71518.00000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,...,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000
mean,4.28913,43.075478,1.430577,15.705025,0.280069,0.103540,0.177829,7.245700,0.424858,0.026511,...,0.398962,65.651864,0.027238,0.180192,0.747938,0.021211,0.023421,0.531684,0.468274,0.000042
std,2.94921,19.952338,1.759864,8.311163,1.068957,0.509187,0.603790,1.994674,0.835638,0.234470,...,0.489688,15.978075,0.162777,0.384350,0.434200,0.144090,0.151236,0.498999,0.498996,0.006477
min,1.00000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.00000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000,0.000000,0.000000,...,0.000000,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.00000,44.000000,1.000000,14.000000,0.000000,0.000000,0.000000,8.000000,0.000000,0.000000,...,0.000000,65.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,6.00000,57.000000,2.000000,20.000000,0.000000,0.000000,0.000000,9.000000,0.000000,0.000000,...,1.000000,75.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000
max,14.00000,132.000000,6.000000,81.000000,42.000000,42.000000,12.000000,16.000000,3.000000,3.000000,...,1.000000,95.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


#### Shuffling and splitting

In [13]:
# randomly shuffling the data
display(df.head())

indices = np.arange(len(df))

shuffled_indices = np.random.permutation(indices)

df = df.iloc[shuffled_indices].reset_index(drop=True)

display(df.head())

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,readmitted,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
0,1,41,0,1,0,0,0,1,0,0,...,0,5.0,0,0,1,0,0,1,0,0
1,3,59,0,18,0,0,0,9,0,0,...,1,15.0,0,0,1,0,0,1,0,0
2,2,11,5,13,2,0,1,6,0,0,...,0,25.0,0,1,0,0,0,1,0,0
3,2,44,1,16,0,0,0,7,0,0,...,0,35.0,0,0,1,0,0,0,1,0
4,1,51,0,8,0,0,0,5,0,0,...,0,45.0,0,0,1,0,0,0,1,0


,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,readmitted,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
0,6,72,2,19,0,0,0,7,0,0,...,1,75.0,0,0,1,0,0,1,0,0
1,8,47,4,9,0,0,0,5,0,0,...,0,65.0,0,1,0,0,0,1,0,0
2,8,69,0,26,0,0,0,9,0,0,...,0,65.0,0,0,1,0,0,1,0,0
3,12,75,3,29,0,0,0,9,0,0,...,0,85.0,0,0,1,0,0,1,0,0
4,3,47,0,18,0,0,0,6,2,0,...,0,55.0,0,0,1,0,0,0,1,0


In [14]:
# splitting X and Y data
X = df.copy().drop(columns=['readmitted'], axis=1)
Y = df.copy()['readmitted']

In [15]:
# splitting the data into train, val, test (60/20/20)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.8, random_state=1234)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, train_size=0.75, random_state=1234)

#### Continuous feature standardizations

In [16]:
# standardizing the continous features between 0 and 1
columns_to_standardize = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
                          'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_mid']

scaler = MinMaxScaler()

X_train[columns_to_standardize] = scaler.fit_transform(X_train[columns_to_standardize])
X_val[columns_to_standardize] = scaler.transform(X_val[columns_to_standardize])
X_test[columns_to_standardize] = scaler.transform(X_test[columns_to_standardize])

print(f"The shape of X_train is {X_train.shape}")
print(f"\nThe shape of Y_train is {Y_train.shape}")
print(f"\nThe shape of X_val is {X_val.shape}")
print(f"\nThe shape of Y_val is {Y_val.shape}")
print(f"\nThe shape of X_test is {X_test.shape}")
print(f"\nThe shape of Y_test is {Y_test.shape}")


The shape of X_train is (42910, 42)

The shape of Y_train is (42910,)

The shape of X_val is (14304, 42)

The shape of Y_val is (14304,)

The shape of X_test is (14304, 42)

The shape of Y_test is (14304,)


In [17]:
display(X_train.head())
display(Y_train.head())

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,diabetesMed,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
14839,0.153846,0.190840,0.500000,0.2250,0.0,0.000000,0.000000,0.266667,0,0,...,1,0.888889,0,0,1,0,0,1,0,0
69258,0.000000,0.045802,0.000000,0.0875,0.0,0.000000,0.000000,0.533333,2,0,...,1,0.555556,0,0,1,0,0,1,0,0
447,0.153846,0.000000,0.166667,0.0250,0.0,0.000000,0.000000,0.466667,0,0,...,0,0.888889,0,0,1,0,0,1,0,0
9126,0.153846,0.458015,0.500000,0.1625,0.0,0.027027,0.166667,0.466667,0,0,...,1,0.777778,0,0,1,0,0,0,1,0
46337,0.615385,0.358779,1.000000,0.1375,0.0,0.000000,0.000000,0.333333,0,0,...,1,0.888889,0,0,1,0,0,0,1,0


14839    1
69258    0
447      0
9126     0
46337    0
Name: readmitted, dtype: int64

## 2. Baseline Model Developments

### 2.1 Majority

### 2.2 Multiclass logistic regression

In [18]:
def build_model(learning_rate=0.01):

  tf.keras.backend.clear_session()

  model = tf.keras.Sequential()

  model.add(keras.Input(shape=(X_train.shape[1],)))

  model.add(keras.layers.Dense(
      units=64,
      activation='relu'
  ))

  model.add(keras.layers.BatchNormalization())
  model.add(keras.layers.Dropout(0.2))


  model.add(keras.layers.Dense(
      units=32,
      activation='relu'
  ))

  model.add(keras.layers.BatchNormalization())
  model.add(keras.layers.Dropout(0.2))


  model.add(keras.layers.Dense(
      units=16,
      activation='relu'
  ))

  # model.add(keras.layers.BatchNormalization())
  # model.add(keras.layers.Dropout(0.2))

  model.add(keras.layers.Dense(
      units=1,
      activation='sigmoid',
      kernel_initializer='glorot_uniform',
      bias_initializer='glorot_uniform'
  ))

  model.compile(
      optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
      loss=keras.losses.BinaryCrossentropy(),
      metrics=['accuracy']
  )

  history = model.fit(
      x=X_train,
      y=Y_train,
      validation_data=(X_val, Y_val),
      batch_size=64,
      epochs=10,
      verbose=1
  )

  return model, history

In [19]:
m1, history = build_model(0.001)

2025-11-26 00:43:48.249206: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-11-26 00:43:48.249317: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-11-26 00:43:48.249327: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2025-11-26 00:43:48.249505: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-26 00:43:48.249514: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/10


2025-11-26 00:43:48.993863: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


671/671 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.5911 - loss: 0.7019 - val_accuracy: 0.5962 - val_loss: 0.6663
Epoch 2/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.6026 - loss: 0.6657 - val_accuracy: 0.5996 - val_loss: 0.6665
Epoch 3/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.6079 - loss: 0.6613 - val_accuracy: 0.6105 - val_loss: 0.6591
Epoch 4/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.6122 - loss: 0.6597 - val_accuracy: 0.6130 - val_loss: 0.6560
Epoch 5/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.6138 - loss: 0.6597 - val_accuracy: 0.6142 - val_loss: 0.6567
Epoch 6/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.6138 - loss: 0.6582 - val_accuracy: 0.6138 - val_loss: 0.6591
Epoch 7/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.6140 - loss: 0.6591 - val_accuracy: 0.6109 - val_loss: 0.6563
Epoch 8/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.6162 - loss: 0.6596 - val_accuracy: 0

In [20]:
preds = (m1.predict(X_val) > 0.5).astype(int)
print(np.unique(preds, return_counts=True))

447/447 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
(array([0, 1]), array([11052,  3252]))


## 3. Notebook Exports

#### 3.1 Export the training, validation, and testing datasets

In [21]:
df_train = pd.concat([X_train, Y_train], axis=1).to_csv('../results/train.csv', index=False)
df_val = pd.concat([X_val, Y_val], axis=1).to_csv('../results/val.csv', index=False)
df_test = pd.concat([X_test, Y_test], axis=1).to_csv('../results/test.csv', index=False)

#### 3.2 Export baseline model(s)

In [22]:
joblib.dump(m1, '../results/baseline.pkl')

['../results/baseline.pkl']

#### 3.3 Export evaluation result(s)

In [23]:
# # Initialize the dataframe
# stats = pd.DataFrame(columns=['feature_importance', 'precision', 'recall',
#                               'false_positive', 'false_negative', 
#                               'final_accuracy_train', 'final_accuracy_test', 'f1'])

# # Insert model statistics from the baseline model
# stats_bsl = pd.DataFrame([{'feature_importance': ,
#                           'precision': ,
#                           'recall': ,
#                           'false_positive' : ,
#                           'false_negative' : ,
#                           'final_accuracy_train' : ,
#                           'final_accuracy_test' : ,
#                           'f1' : }])

# # Combine the stats and export as csv
# stats = pd.concat([stats, stats_bsl], ignore_index=True)
# stats.to_csv('../results/stats_baseline.csv', index=False) 